# Coastal flood step 04: maps and charts (maximum_scenario)

Generates sign-based maps and summary charts for this set scenario.


In [ ]:
import geopandas
import numpy
import pandas
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import sys
import pathlib

robyn_libraries_path = pathlib.Path("../../../robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

# !{sys.executable} -m pip install "nismod-snail==0.5.3"


In [ ]:
# processed_data_path = 'L:\Jamaica\Inputs'
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_path = base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario"
intersections_path = base_path / "dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
jamaica_metric_grid_crs = "EPSG:3448"

# base_path = 'Z:\\jamaica\\Inputs'
# output_path = 'Z:\\jamaica\\Results'


In [ ]:
# network_csv = os.path.join(processed_data_path,
#                             "networks",
#                             "network_layers_hazard_intersections_details.csv")
# hazard_csv = os.path.join(processed_data_path,
#                             "coastal_flood_rasters.csv")
# damage_curves_csv = os.path.join(processed_data_path,
#                             "damage_curves",
# #                             "asset_damage_curve_mapping.csv")
# hazard_damage_parameters_csv = os.path.join(processed_data_path,
#                             "damage_curves",
#                             "hazard_damage_parameters.csv")
# damage_results_folder = "direct_damages"


In [ ]:
# Shared map inputs (loaded once)
jamaica_boundary_path = data_root / "boundaries/jamaica.gpkg"
airport_damage_file = Path(output_path) / "damage_estimates" / "airport_polygon_areas_asset_damages_groupedby.gpkg"

if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f"Missing Jamaica boundary file: {jamaica_boundary_path}")
jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)

if airport_damage_file.exists():
    airport_damage_base = geopandas.read_file(airport_damage_file).to_crs(jamaica_metric_grid_crs)
else:
    airport_damage_base = None


In [ ]:

# Draw roads using split coastal geometry, but color by edge-level avoided damages (J$).
roads_damage_file = Path(output_path) / "damage_estimates" / "roads_edges_asset_damages_groupedby.gpkg"
roads_splits_file = Path(intersections_path) / "roads_splits__coastal_flood_rasters_for_intersections__edges.geoparquet"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not roads_damage_file.exists():
    raise FileNotFoundError(f"Missing roads damage file: {roads_damage_file}")
if not roads_splits_file.exists():
    raise FileNotFoundError(f"Missing roads splits file: {roads_splits_file}")
roads_damage = geopandas.read_file(roads_damage_file)[["edge_id", avoided_damage_column]].copy()
roads_splits = geopandas.read_parquet(roads_splits_file)[["edge_id", "geometry"]].copy()
roads_splits = roads_splits.to_crs(jamaica_metric_grid_crs)

roads_segments = roads_splits.merge(roads_damage, on="edge_id", how="left")
roads_segments[avoided_damage_column] = roads_segments[avoided_damage_column].fillna(0.0)

zero_value_roads = roads_segments[numpy.abs(roads_segments[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_roads = roads_segments[roads_segments[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_roads = roads_segments[roads_segments[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_roads)} split road segments")
print(f"Negative avoided damages (red): {len(negative_value_roads)} split road segments")
print(f"Zero (white): {len(zero_value_roads)} split road segments")
if not roads_segments.empty:
    print(f"Min value: {roads_segments[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {roads_segments[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(11, 10))
axis.set_facecolor("#ffffff")
jamaica_boundary.boundary.plot(ax=axis, color="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_roads.empty:
    zero_value_roads.plot(ax=axis, color="#ffffff", linewidth=0.45, alpha=0.45, zorder=2)
if not positive_value_roads.empty:
    positive_value_roads.plot(ax=axis, color="#0b8f3f", linewidth=1.6, alpha=0.95, zorder=3)
if not negative_value_roads.empty:
    negative_value_roads.plot(ax=axis, color="#c81e1e", linewidth=1.6, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], color="#0b8f3f", lw=2.2, label="Avoided damages > 0 (green)"),
    Line2D([0], [0], color="#c81e1e", lw=2.2, label="Damages increase < 0 (red)"),
    Line2D([0], [0], color="#ffffff", lw=2.2, label="Zero change (white)"),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
axis.set_title(f"Road coastal-segment damage change (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"roads_coastal_segment_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

top_positive = roads_damage[["edge_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False).head(10)
top_negative = roads_damage[["edge_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=True).head(10)
print("\nTop 10 positive avoided damages (road edges):")
display(top_positive)
print("\nTop 10 negative avoided damages (road edges):")
display(top_negative)


In [ ]:

return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if airport_damage_base is None:
    raise FileNotFoundError(f"Missing airport damage file: {airport_damage_file}")
airport_damage = airport_damage_base.copy()
if avoided_damage_column not in airport_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

airport_damage = airport_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_airports = airport_damage[numpy.abs(airport_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_airports = airport_damage[airport_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_airports = airport_damage[airport_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_airports)} airports")
print(f"Negative avoided damages (red): {len(negative_value_airports)} airports")
print(f"Zero (white): {len(zero_value_airports)} airports")
if not airport_damage.empty:
    print(f"Min value: {airport_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {airport_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_airports.empty:
    zero_value_airports.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, zorder=2)
if not positive_value_airports.empty:
    positive_value_airports.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, zorder=3)
if not negative_value_airports.empty:
    negative_value_airports.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=12, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=12, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=12, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
axis.set_title(f"Airport damage change (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"airports_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(airport_damage[["node_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))



In [ ]:

port_damage_file = Path(output_path) / "damage_estimates" / "port_polygon_areas_asset_damages_groupedby.gpkg"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not port_damage_file.exists():
    raise FileNotFoundError(f"Missing port damage file: {port_damage_file}")
port_damage = geopandas.read_file(port_damage_file)
if avoided_damage_column not in port_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

port_damage = port_damage.to_crs(jamaica_metric_grid_crs)
port_damage = port_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_ports = port_damage[numpy.abs(port_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_ports = port_damage[port_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_ports = port_damage[port_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_ports)} port polygons")
print(f"Negative avoided damages (red): {len(negative_value_ports)} port polygons")
print(f"Zero (white): {len(zero_value_ports)} port polygons")
if not port_damage.empty:
    print(f"Min value: {port_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {port_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_ports.empty:
    zero_value_ports.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, zorder=2)
if not positive_value_ports.empty:
    positive_value_ports.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, zorder=3)
if not negative_value_ports.empty:
    negative_value_ports.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=12, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=12, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=12, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
axis.set_title(f"Port damage change (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"ports_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(port_damage[["node_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))


In [ ]:

energy_nodes_damage_file = Path(output_path) / "damage_estimates" / "electricity_network_v3.1_nodes_asset_damages_groupedby.gpkg"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not energy_nodes_damage_file.exists():
    raise FileNotFoundError(f"Missing energy nodes damage file: {energy_nodes_damage_file}")
energy_nodes = geopandas.read_file(energy_nodes_damage_file)
if avoided_damage_column not in energy_nodes.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

energy_nodes = energy_nodes.to_crs(jamaica_metric_grid_crs)
energy_nodes = energy_nodes.dropna(subset=[avoided_damage_column]).copy()

zero_value_nodes = energy_nodes[numpy.abs(energy_nodes[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_nodes = energy_nodes[energy_nodes[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_nodes = energy_nodes[energy_nodes[avoided_damage_column] < -zero_tolerance_jd].copy()


print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_nodes)} energy nodes")
print(f"Negative avoided damages (red): {len(negative_value_nodes)} energy nodes")
print(f"Zero (white): {len(zero_value_nodes)} energy nodes")
if not energy_nodes.empty:
    print(f"Min value: {energy_nodes[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {energy_nodes[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_nodes.empty:
    zero_value_nodes.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, markersize=55, zorder=2)
if not positive_value_nodes.empty:
    positive_value_nodes.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, markersize=70, zorder=3)
if not negative_value_nodes.empty:
    negative_value_nodes.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, markersize=70, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=10, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=10, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=10, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
axis.set_title(f"Energy (nodes) damage change (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"energy_nodes_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(energy_nodes[["id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))



In [ ]:

buildings_damage_file = Path(output_path) / "damage_estimates" / "buildings_assigned_economic_activity_areas_asset_damages_groupedby.gpkg"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not buildings_damage_file.exists():
    raise FileNotFoundError(f"Missing buildings damage file: {buildings_damage_file}")
buildings_damage = geopandas.read_file(buildings_damage_file)
if avoided_damage_column not in buildings_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

buildings_damage = buildings_damage.to_crs(jamaica_metric_grid_crs)
buildings_damage = buildings_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_buildings = buildings_damage[numpy.abs(buildings_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_buildings = buildings_damage[buildings_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_buildings = buildings_damage[buildings_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_buildings)} buildings")
print(f"Negative avoided damages (red): {len(negative_value_buildings)} buildings")
print(f"Zero (white): {len(zero_value_buildings)} buildings")
if not buildings_damage.empty:
    print(f"Min value: {buildings_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {buildings_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_buildings.empty:
    zero_value_buildings.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.2, alpha=0.65, zorder=2)
if not positive_value_buildings.empty:
    positive_value_buildings.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=0.2, alpha=0.88, zorder=3)
if not negative_value_buildings.empty:
    negative_value_buildings.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=0.2, alpha=0.88, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=10, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=10, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=10, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6,
              label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
axis.set_title(f"Buildings damage change (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"buildings_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(buildings_damage[["osm_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False).head(30))


In [ ]:
# Additional subsector sign maps so each remaining layer has its own map.
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

additional_subsector_map_settings = [
    {
        "layer_label": "Rail edges",
        "damage_file_path": Path(output_path) / "damage_estimates" / "rail_edges_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "line",
        "identifier_column_name": "edge_id",
        "output_file_stem": "rail_edges_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Rail nodes",
        "damage_file_path": Path(output_path) / "damage_estimates" / "rail_nodes_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "point",
        "identifier_column_name": "node_id",
        "output_file_stem": "rail_nodes_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Energy edges",
        "damage_file_path": Path(output_path) / "damage_estimates" / "electricity_network_v3.1_edges_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "line",
        "identifier_column_name": "id",
        "output_file_stem": "energy_edges_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Pipelines (NWC) edges",
        "damage_file_path": Path(output_path) / "damage_estimates" / "pipelines_NWC_edges_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "line",
        "identifier_column_name": "edge_id",
        "output_file_stem": "pipelines_nwc_edges_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Potable facilities (NWC) nodes",
        "damage_file_path": Path(output_path) / "damage_estimates" / "potable_facilities_NWC_nodes_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "point",
        "identifier_column_name": "node_id",
        "output_file_stem": "potable_facilities_nwc_nodes_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Wastewater facilities (NWC) nodes",
        "damage_file_path": Path(output_path) / "damage_estimates" / "waste_water_facilities_NWC_nodes_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "point",
        "identifier_column_name": "node_id",
        "output_file_stem": "wastewater_facilities_nwc_nodes_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Irrigation assets (NIC) edges",
        "damage_file_path": Path(output_path) / "damage_estimates" / "irrigation_assets_NIC_edges_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "line",
        "identifier_column_name": "edge_id",
        "output_file_stem": "irrigation_assets_nic_edges_damage_change_sign_map_with_boundary",
    },
    {
        "layer_label": "Irrigation assets (NIC) nodes",
        "damage_file_path": Path(output_path) / "damage_estimates" / "irrigation_assets_NIC_nodes_asset_damages_groupedby.gpkg",
        "map_geometry_kind": "point",
        "identifier_column_name": "node_id",
        "output_file_stem": "irrigation_assets_nic_nodes_damage_change_sign_map_with_boundary",
    },
]

for layer_map_setting in additional_subsector_map_settings:
    layer_label = layer_map_setting["layer_label"]
    damage_file_path = layer_map_setting["damage_file_path"]
    map_geometry_kind = layer_map_setting["map_geometry_kind"]
    identifier_column_name = layer_map_setting["identifier_column_name"]
    output_file_stem = layer_map_setting["output_file_stem"]

    if not damage_file_path.exists():
        print(f"Missing damage file, skipping {layer_label}: {damage_file_path}")
        continue

    layer_damage = geopandas.read_file(damage_file_path)
    if avoided_damage_column not in layer_damage.columns:
        print(f"Missing column {avoided_damage_column} in {damage_file_path.name}; skipping {layer_label}.")
        continue

    layer_damage = layer_damage.to_crs(jamaica_metric_grid_crs)
    layer_damage = layer_damage.dropna(subset=[avoided_damage_column]).copy()

    zero_value_assets = layer_damage[numpy.abs(layer_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
    positive_value_assets = layer_damage[layer_damage[avoided_damage_column] > zero_tolerance_jd].copy()
    negative_value_assets = layer_damage[layer_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

    print(f"\n{layer_label} | RP {return_period_years}")
    print(f"Positive avoided damages (green): {len(positive_value_assets)} assets")
    print(f"Negative avoided damages (red): {len(negative_value_assets)} assets")
    print(f"Zero (white): {len(zero_value_assets)} assets")
    if not layer_damage.empty:
        print(f"Min value: {layer_damage[avoided_damage_column].min():,.2f} J$")
        print(f"Max value: {layer_damage[avoided_damage_column].max():,.2f} J$")

    figure, axis = plt.subplots(figsize=(10, 9))
    axis.set_facecolor("#ffffff")
    jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

    if map_geometry_kind == "line":
        if not zero_value_assets.empty:
            zero_value_assets.plot(ax=axis, color="#ffffff", linewidth=1.1, alpha=0.7, zorder=2)
        if not positive_value_assets.empty:
            positive_value_assets.plot(ax=axis, color="#0b8f3f", linewidth=1.4, alpha=0.95, zorder=3)
        if not negative_value_assets.empty:
            negative_value_assets.plot(ax=axis, color="#c81e1e", linewidth=1.4, alpha=0.95, zorder=4)

        legend_handles = [
            Line2D([0], [0], color="#0b8f3f", lw=2.2, label="Avoided damages > 0 (green)"),
            Line2D([0], [0], color="#c81e1e", lw=2.2, label="Damages increase < 0 (red)"),
            Line2D([0], [0], color="#ffffff", lw=2.2, label="Zero change (white)"),
        ]
    elif map_geometry_kind == "point":
        if not zero_value_assets.empty:
            zero_value_assets.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.8, markersize=55, zorder=2)
        if not positive_value_assets.empty:
            positive_value_assets.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.0, alpha=0.95, markersize=70, zorder=3)
        if not negative_value_assets.empty:
            negative_value_assets.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.0, alpha=0.95, markersize=70, zorder=4)

        legend_handles = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=10, label='Avoided damages > 0 (green)'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=10, label='Damages increase < 0 (red)'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=10, label='Zero change (white)'),
        ]
    else:
        if not zero_value_assets.empty:
            zero_value_assets.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.5, alpha=0.75, zorder=2)
        if not positive_value_assets.empty:
            positive_value_assets.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=0.6, alpha=0.95, zorder=3)
        if not negative_value_assets.empty:
            negative_value_assets.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=0.6, alpha=0.95, zorder=4)

        legend_handles = [
            Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=10, label='Avoided damages > 0 (green)'),
            Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=10, label='Damages increase < 0 (red)'),
            Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=10, label='Zero change (white)'),
        ]

    axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")
    Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
    axis.set_title(f"{layer_label} damage change (RP {return_period_years})", fontsize=13)
    axis.set_axis_off()
    plt.tight_layout()

    map_image_file = Path(output_path) / "damage_estimates" / f"{output_file_stem}_rp_{return_period_years}.png"
    figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
    print(f"Saved map image: {map_image_file}")
    plt.show()

    if identifier_column_name in layer_damage.columns and not layer_damage.empty:
        display(layer_damage[[identifier_column_name, avoided_damage_column]].sort_values(avoided_damage_column, ascending=False).head(20))




In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file)
sector_totals = (
    summary.groupby(["Sector", "ReturnPeriod"], as_index=False)["Avoided_Damages_JD"]
    .sum()
)

jd_per_usd = 150.0
sector_totals["Avoided_Damages_USD"] = sector_totals["Avoided_Damages_JD"] / jd_per_usd

sector_order = ["buildings", "energy", "transport", "water"]
return_periods = [25, 100, 500]

def currency_formatter(x, _):
    return f"{x:,.0f}"

for rp in return_periods:
    rp_data = sector_totals[sector_totals["ReturnPeriod"] == rp].copy()
    rp_data["Sector"] = pandas.Categorical(rp_data["Sector"], categories=sector_order, ordered=True)
    rp_data = rp_data.sort_values("Sector")

    bar_colors = ["#0b8f3f" if value >= 0 else "#c81e1e" for value in rp_data["Avoided_Damages_USD"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(rp_data["Sector"], rp_data["Avoided_Damages_USD"], color=bar_colors, edgecolor="#2f2f2f", linewidth=0.6)
    ax.axhline(0, color="#7f7f7f", linewidth=0.8)
    ax.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
    ax.set_xlabel("Sector")
    ax.set_ylabel("Avoided damages (US$)")
    ax.set_title(f"Total sector avoided damages (RP {rp})")

    for bar, value in zip(bars, rp_data["Avoided_Damages_USD"]):
        y_pos = value if value >= 0 else value
        va = "bottom" if value >= 0 else "top"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            f"{value:,.0f}",
            ha="center",
            va=va,
            fontsize=9,
            rotation=0,
        )

    plt.tight_layout()
    output_chart = Path(output_path) / "damage_estimates" / f"sector_avoided_damages_bar_rp_{rp}.png"
    fig.savefig(output_chart, dpi=300, bbox_inches="tight")
    print(f"Saved: {output_chart}")
    plt.show()



In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file)
subsector_totals = (
    summary.groupby(["Sector", "Subsector", "ReturnPeriod"], as_index=False)["Avoided_Damages_JD"]
    .sum()
)

jd_per_usd = 150.0
subsector_totals["Avoided_Damages_USD"] = subsector_totals["Avoided_Damages_JD"] / jd_per_usd

target_sectors = ["transport", "water"]
return_periods = [25, 100, 500]

def currency_formatter(x, _):
    return f"{x:,.0f}"

for sector_name in target_sectors:
    sector_data = subsector_totals[subsector_totals["Sector"] == sector_name].copy()
    subsector_order = sorted(sector_data["Subsector"].unique().tolist())

    for rp in return_periods:
        rp_data = sector_data[sector_data["ReturnPeriod"] == rp].copy()
        rp_data["Subsector"] = pandas.Categorical(rp_data["Subsector"], categories=subsector_order, ordered=True)
        rp_data = rp_data.sort_values("Subsector")

        bar_colors = ["#0b8f3f" if value >= 0 else "#c81e1e" for value in rp_data["Avoided_Damages_USD"]]

        fig, ax = plt.subplots(figsize=(11, 5.5))
        bars = ax.bar(rp_data["Subsector"], rp_data["Avoided_Damages_USD"], color=bar_colors, edgecolor="#2f2f2f", linewidth=0.6)
        ax.axhline(0, color="#7f7f7f", linewidth=0.8)
        ax.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax.set_xlabel("Subsector")
        ax.set_ylabel("Avoided damages (US$)")
        ax.set_title(f"{sector_name.capitalize()} subsector avoided damages (RP {rp})")
        plt.xticks(rotation=25, ha="right")

        for bar, value in zip(bars, rp_data["Avoided_Damages_USD"]):
            va = "bottom" if value >= 0 else "top"
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value,
                f"{value:,.0f}",
                ha="center",
                va=va,
                fontsize=8,
            )

        plt.tight_layout()
        output_chart = Path(output_path) / "damage_estimates" / f"{sector_name}_subsector_avoided_damages_bar_rp_{rp}.png"
        fig.savefig(output_chart, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_chart}")
        plt.show()



In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file).copy()

# Row-level share for each sector/subsector/return period record
summary["Avoided_Share_of_Baseline"] = numpy.where(
    summary["Damages_Without_Mangroves_JD"] > 0,
    summary["Avoided_Damages_JD"] / summary["Damages_Without_Mangroves_JD"],
    numpy.nan,
)
summary["Avoided_Share_of_Baseline_pct"] = 100 * summary["Avoided_Share_of_Baseline"]

summary_display_cols = [
    "Sector",
    "Subsector",
    "ReturnPeriod",
    "Damages_With_Mangroves_JD",
    "Damages_Without_Mangroves_JD",
    "Avoided_Damages_JD",
    "Avoided_Share_of_Baseline_pct",
]

print("Subsector-level avoided damages as % of baseline:")
display(
    summary[summary_display_cols]
    .sort_values(["Sector", "Subsector", "ReturnPeriod"])
    .round({"Avoided_Share_of_Baseline_pct": 2})
)

# Sector-level share uses ratio-of-sums (preferred aggregation)
sector_share = (
    summary.groupby(["Sector", "ReturnPeriod"], as_index=False)[
        ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD"]
    ]
    .sum()
)
sector_share["Avoided_Share_of_Baseline"] = numpy.where(
    sector_share["Damages_Without_Mangroves_JD"] > 0,
    sector_share["Avoided_Damages_JD"] / sector_share["Damages_Without_Mangroves_JD"],
    numpy.nan,
)
sector_share["Avoided_Share_of_Baseline_pct"] = 100 * sector_share["Avoided_Share_of_Baseline"]

print("Sector-level avoided damages as % of baseline (ratio-of-sums):")
display(
    sector_share[
        [
            "Sector",
            "ReturnPeriod",
            "Damages_Without_Mangroves_JD",
            "Avoided_Damages_JD",
            "Avoided_Share_of_Baseline_pct",
        ]
    ]
    .sort_values(["Sector", "ReturnPeriod"])
    .round({"Avoided_Share_of_Baseline_pct": 2})
)

print("Quick comparison table (sector x return period, %):")
display(
    sector_share.pivot(index="Sector", columns="ReturnPeriod", values="Avoided_Share_of_Baseline_pct").round(2)
)

out_dir = Path(output_path) / "damage_estimates"
subsector_out = out_dir / "sector_subsector_return_period_damages_with_avoided_share.csv"
sector_out = out_dir / "sector_return_period_avoided_share.csv"

summary.to_csv(subsector_out, index=False)
sector_share.to_csv(sector_out, index=False)

print(f"Saved: {subsector_out}")
print(f"Saved: {sector_out}")


In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file)
sector_share = (
    summary.groupby(["Sector", "ReturnPeriod"], as_index=False)[
        ["Damages_Without_Mangroves_JD", "Avoided_Damages_JD"]
    ]
    .sum()
)

sector_share["Avoided_Share_of_Baseline"] = numpy.where(
    sector_share["Damages_Without_Mangroves_JD"] > 0,
    sector_share["Avoided_Damages_JD"] / sector_share["Damages_Without_Mangroves_JD"],
    numpy.nan,
)
sector_share["Avoided_Share_of_Baseline_pct"] = 100 * sector_share["Avoided_Share_of_Baseline"]

sector_order = ["buildings", "energy", "transport", "water"]
return_periods = [25, 100, 500]

def pct_formatter(x, _):
    return f"{x:.0f}%"

for rp in return_periods:
    rp_data = sector_share[sector_share["ReturnPeriod"] == rp].copy()
    rp_data["Sector"] = pandas.Categorical(rp_data["Sector"], categories=sector_order, ordered=True)
    rp_data = rp_data.sort_values("Sector")

    bar_values = rp_data["Avoided_Share_of_Baseline_pct"].fillna(0)
    bar_colors = ["#0b8f3f" if value >= 0 else "#c81e1e" for value in bar_values]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(rp_data["Sector"], bar_values, color=bar_colors, edgecolor="#2f2f2f", linewidth=0.6)
    ax.axhline(0, color="#7f7f7f", linewidth=0.8)
    ax.yaxis.set_major_formatter(FuncFormatter(pct_formatter))
    ax.set_xlabel("Sector")
    ax.set_ylabel("Avoided damages (% of baseline)")
    ax.set_title(f"Sector avoided damages as % of baseline (RP {rp})")
    ax.set_ylim(0, 100)

    for bar, value in zip(bars, rp_data["Avoided_Share_of_Baseline_pct"]):
        if pandas.isna(value):
            label = "NA"
            y_pos = 0
            va = "bottom"
        else:
            label = f"{value:.0f}%"
            if value >= 99:
                y_pos = 99
                va = "top"
            else:
                y_pos = value
                va = "bottom" if value >= 0 else "top"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            label,
            ha="center",
            va=va,
            fontsize=9,
        )

    plt.tight_layout()
    output_chart = Path(output_path) / "damage_estimates" / f"sector_avoided_share_bar_rp_{rp}.png"
    fig.savefig(output_chart, dpi=300, bbox_inches="tight")
    print(f"Saved: {output_chart}")
    plt.show()


